In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("Delta-Iceberg-MinIO")
    .master("local[*]")
    .config("spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension,"
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg.type", "hadoop")
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://warehouse/iceberg/")
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true")
    .config("spark.sql.catalog.iceberg.client.region", "us-east-1")
    .config("spark.sql.catalog.iceberg.s3.access-key-id", "matrix")
    .config("spark.sql.catalog.iceberg.s3.secret-access-key", "matrix123")
    .getOrCreate()
)


In [3]:
hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://minio:9000")
hconf.set("fs.s3a.access.key", "matrix")
hconf.set("fs.s3a.secret.key", "matrix123")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "false")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")


In [4]:
raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/home/jovyan/transactions_raw .csv")
)

raw_df.show(5)
raw_df.printSchema()

+-----+-----------+----------+------+--------+---------+
|   id|customer_id|        dt|amount|currency|   status|
+-----+-----------+----------+------+--------+---------+
|39329|        247|2025-03-26|163.99|     AZN|completed|
|41803|        685|2025-03-24|410.52|     EUR|completed|
| 7150|        385|2025-03-02|387.52|     AZN|completed|
|20125|        726|2025-02-15|221.73|     EUR|  pending|
|40951|        219|2025-02-05|581.89|     AZN|completed|
+-----+-----------+----------+------+--------+---------+
only showing top 5 rows

root
 |-- id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- dt: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)



In [24]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.bronze")

DELTA_BRONZE   = "s3a://warehouse/delta/bronze/transactions"
ICEBERG_BRONZE = "iceberg.bronze.transactions"

raw_df.write.format("delta").mode("overwrite").save(DELTA_BRONZE)
print(f"Delta Bronze yazıldı")

spark.sql(f"DROP TABLE IF EXISTS {ICEBERG_BRONZE}")
raw_df.writeTo(ICEBERG_BRONZE).createOrReplace()
print(f"Iceberg Bronze yazıldı")

Delta Bronze yazıldı
Iceberg Bronze yazıldı


In [5]:
raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/home/jovyan/transactions_raw .csv")
)

In [6]:
clean_df = (

    raw_df

    .filter(F.col("id").isNotNull())

    .filter(F.col("customer_id").isNotNull())

    .filter(F.col("dt").isNotNull()) 

    .filter(F.col("amount").isNotNull()) 

    .dropDuplicates(["id"])

    .withColumn("id", F.col("id").cast("integer"))
    
    .withColumn("customer_id", F.col("customer_id").cast("integer"))
    
    .withColumn("amount", F.col("amount").cast("double"))
    
    .withColumn("dt", F.to_date(F.col("dt"), "yyyy-MM-dd"))
    
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    
    .withColumn("status", F.lower(F.trim(F.col("status"))))
    
    .orderBy("id")

) 



In [7]:
for col_name, data_type in clean_df.dtypes:
    if data_type == "string":
        clean_df = clean_df.withColumn(
            col_name,
            F.when((F.trim(F.col(col_name)) == "") | (F.col(col_name).isNull()), None)
            .otherwise(F.trim(F.col(col_name)))
        )

In [9]:
print("Raw sətir sayı:", raw_df.count())

print("Clean sətir sayı:", clean_df.count())

Raw sətir sayı: 52100
Clean sətir sayı: 50600


In [18]:
DELTA_SILVER = "s3a://warehouse/delta/silver"

clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(DELTA_SILVER)

In [11]:
ICEBERG_SILVER = "iceberg.silver.transactions"

spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.silver")

spark.sql(f"DROP TABLE IF EXISTS {ICEBERG_SILVER}")

clean_df.writeTo(ICEBERG_SILVER).createOrReplace()

In [20]:
print(f"Bronze sətir sayı: {raw_df.count()}")
print(f"Silver sətir sayı: {clean_df.count()}")
print(f"Silinən sətrlərin sayı: {raw_df.count() - clean_df.count()}")

Bronze sətir sayı: 52100
Silver sətir sayı: 50600
Silinən sətrlərin sayı: 1500


In [21]:
bronze_df = raw_df
silver_df = clean_df

In [22]:
from pyspark.sql import functions as F

gold_df = (
    clean_df
    .groupBy("dt", "currency")
    .agg(
        F.count("id").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount"),
        F.min("amount").alias("min_amount"),
        F.max("amount").alias("max_amount")
    )
    .orderBy("dt", "currency")
)

gold_df.show()

+----------+--------+------------------+------------------+------------------+----------+----------+
|        dt|currency|total_transactions|      total_amount|        avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------------+------------------+----------+----------+
|2025-01-01|     AZN|               338| 79560.50999999998|235.38612426035496|    -208.6|   1037.59|
|2025-01-01|     EUR|                66|14668.710000000001|222.25318181818184|     16.44|    688.98|
|2025-01-01|     USD|               137|30555.190000000002|223.03058394160587|   -253.76|    771.66|
|2025-01-02|     AZN|               335| 77816.69999999998|232.28865671641785|   -243.21|   1020.34|
|2025-01-02|     EUR|                82|19577.050000000007|238.74451219512204|     10.28|    677.83|
|2025-01-02|     USD|               139|27819.179999999986| 200.1379856115107|    -74.83|    655.42|
|2025-01-03|     AZN|               354| 81370.61999999998|229.86050847457622|   -215.68|  

In [23]:
DELTA_GOLD = "s3a://warehouse/delta/gold/transaction_summary"

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(DELTA_GOLD)

In [24]:
ICEBERG_GOLD = "iceberg.gold.transaction_summary"

gold_df.writeTo(ICEBERG_GOLD) \
    .using("iceberg") \
    .createOrReplace()

In [34]:
from pyspark.sql import functions as F

insert_data = [
    ("2026-09-07", "AZN", 10, 1500.00, 150.00, 20.00, 500.00)
]

insert_df = spark.createDataFrame(
    insert_data,
    [
        "dt",
        "currency",
        "total_transactions",
        "total_amount",
        "avg_amount",
        "min_amount",
        "max_amount"
    ]
)


fixed_insert_df = insert_df.withColumn(
    "dt",
    F.to_date("dt")
)

fixed_insert_df.write \
    .format("delta") \
    .mode("append") \
    .save(DELTA_GOLD)

spark.read \
    .format("delta") \
    .load(DELTA_GOLD) \
    .filter("dt = '2026-09-07' AND currency = 'AZN'") \
    .show()

+----------+--------+------------------+------------+----------+----------+----------+
|        dt|currency|total_transactions|total_amount|avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------+----------+----------+----------+
|2026-09-07|     AZN|                10|      1500.0|     150.0|      20.0|     500.0|
+----------+--------+------------------+------------+----------+----------+----------+



In [36]:
fixed_insert_df.writeTo(ICEBERG_GOLD).append()

print("ICEBERG - INSERT NƏTİCƏSİ")

spark.table(ICEBERG_GOLD) \
    .filter("dt = '2026-09-07' AND currency = 'AZN'") \
    .show()

ICEBERG - INSERT NƏTİCƏSİ
+----------+--------+------------------+------------+----------+----------+----------+
|        dt|currency|total_transactions|total_amount|avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------+----------+----------+----------+
|2026-09-07|     AZN|                10|      1500.0|     150.0|      20.0|     500.0|
+----------+--------+------------------+------------+----------+----------+----------+



In [37]:
from delta.tables import DeltaTable


delta_table = DeltaTable.forPath(
    spark,
    DELTA_GOLD
)

delta_table.update(
    condition="""
        dt = '2026-09-07'
        AND currency = 'AZN'
    """,
    set={
        "total_transactions": "20",
        "total_amount": "3000.00",
        "avg_amount": "150.00",
        "min_amount": "30.00",
        "max_amount": "700.00"
    }
)

print("DELTA - UPDATE NƏTİCƏSİ")

spark.read \
    .format("delta") \
    .load(DELTA_GOLD) \
    .filter("dt = '2026-09-07' AND currency = 'AZN'") \
    .show()

spark.sql("""
    UPDATE iceberg.gold.transaction_summary
    SET
        total_transactions = 20,
        total_amount = 3000.00,
        avg_amount = 150.00,
        min_amount = 30.00,
        max_amount = 700.00
    WHERE dt = '2026-09-07'
      AND currency = 'AZN'
""")

print("ICEBERG - UPDATE NƏTİCƏSİ")

spark.table(ICEBERG_GOLD) \
    .filter("dt = '2026-09-07' AND currency = 'AZN'") \
    .show()

DELTA - UPDATE NƏTİCƏSİ
+----------+--------+------------------+------------+----------+----------+----------+
|        dt|currency|total_transactions|total_amount|avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------+----------+----------+----------+
|2026-09-07|     AZN|                20|      3000.0|     150.0|      30.0|     700.0|
+----------+--------+------------------+------------+----------+----------+----------+

ICEBERG - UPDATE NƏTİCƏSİ
+----------+--------+------------------+------------+----------+----------+----------+
|        dt|currency|total_transactions|total_amount|avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------+----------+----------+----------+
|2026-09-07|     AZN|                20|      3000.0|     150.0|      30.0|     700.0|
+----------+--------+------------------+------------+----------+----------+----------+



In [38]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    DELTA_GOLD
)

delta_table.delete("""
    dt = '2026-09-07'
    AND currency = 'AZN'
""")

print("DELTA - DELETE NƏTİCƏSİ")

spark.read \
    .format("delta") \
    .load(DELTA_GOLD) \
    .filter("dt = '2026-09-07'") \
    .show()


spark.sql("""
    DELETE FROM iceberg.gold.transaction_summary
    WHERE dt = '2026-09-07'
      AND currency = 'AZN'
""")

print("ICEBERG - DELETE NƏTİCƏSİ")

spark.table(ICEBERG_GOLD) \
    .filter("dt = '2026-09-07'") \
    .show()

DELTA - DELETE NƏTİCƏSİ
+---+--------+------------------+------------+----------+----------+----------+
| dt|currency|total_transactions|total_amount|avg_amount|min_amount|max_amount|
+---+--------+------------------+------------+----------+----------+----------+
+---+--------+------------------+------------+----------+----------+----------+

ICEBERG - DELETE NƏTİCƏSİ
+---+--------+------------------+------------+----------+----------+----------+
| dt|currency|total_transactions|total_amount|avg_amount|min_amount|max_amount|
+---+--------+------------------+------------+----------+----------+----------+
+---+--------+------------------+------------+----------+----------+----------+



In [39]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


merge_data = [
    ("2026-09-07", "AZN", 20, 3000.00, 150.00, 30.00, 700.00),
    ("2026-09-07", "USD", 15, 2500.00, 166.67, 20.00, 600.00)
]

merge_df = spark.createDataFrame(
    merge_data,
    [
        "dt",
        "currency",
        "total_transactions",
        "total_amount",
        "avg_amount",
        "min_amount",
        "max_amount"
    ]
)


merge_df = merge_df.withColumn(
    "dt",
    F.to_date("dt")
)



delta_table = DeltaTable.forPath(
    spark,
    DELTA_GOLD
)

delta_table.alias("target") \
    .merge(
        merge_df.alias("source"),
        """
        target.dt = source.dt
        AND target.currency = source.currency
        """
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("DELTA - MERGE / UPSERT NƏTİCƏSİ")

spark.read \
    .format("delta") \
    .load(DELTA_GOLD) \
    .filter("dt = '2026-09-07'") \
    .orderBy("currency") \
    .show()



merge_df.createOrReplaceTempView("merge_source")

spark.sql("""
    MERGE INTO iceberg.gold.transaction_summary AS target
    USING merge_source AS source

    ON target.dt = source.dt
       AND target.currency = source.currency

    WHEN MATCHED THEN
        UPDATE SET *

    WHEN NOT MATCHED THEN
        INSERT *
""")

print("ICEBERG - MERGE / UPSERT NƏTİCƏSİ")

spark.table(ICEBERG_GOLD) \
    .filter("dt = '2026-09-07'") \
    .orderBy("currency") \
    .show()

DELTA - MERGE / UPSERT NƏTİCƏSİ
+----------+--------+------------------+------------+----------+----------+----------+
|        dt|currency|total_transactions|total_amount|avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------+----------+----------+----------+
|2026-09-07|     AZN|                20|      3000.0|     150.0|      30.0|     700.0|
|2026-09-07|     USD|                15|      2500.0|    166.67|      20.0|     600.0|
+----------+--------+------------------+------------+----------+----------+----------+

ICEBERG - MERGE / UPSERT NƏTİCƏSİ
+----------+--------+------------------+------------+----------+----------+----------+
|        dt|currency|total_transactions|total_amount|avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------+----------+----------+----------+
|2026-09-07|     AZN|                20|      3000.0|     150.0|      30.0|     700.0|
|2026-09-07|     USD|                15|      2500.0|    166.67

In [41]:
# Delta history 

delta_history = spark.sql(f"""
    DESCRIBE HISTORY delta.`{DELTA_GOLD}`
""")

delta_history.select(
    "version",
    "timestamp",
    "operation",
    "operationParameters"
).show(truncate=False)


+-------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                                                                                                                                                                |
+-------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|4      |2026-09-07 08:26:00|MERGE    |{predicate -> ["((dt#5992 = dt#5984) AND (currency#5993 = currency#5971))"], matchedPredicates -> [{"actionType":"update"}], notMatchedPredicates -> [{"actionType":"insert"}], notMatchedBySourcePredicates 

In [42]:
# Delta versions

version_2 = spark.read \
    .format("delta") \
    .option("versionAsOf", 2) \
    .load(DELTA_GOLD)

version_4 = spark.read \
    .format("delta") \
    .option("versionAsOf", 4) \
    .load(DELTA_GOLD)

print("VERSION 2:")
version_2.show()

print("VERSION 4:")
version_4.show()

VERSION 2:
+----------+--------+------------------+------------------+------------------+----------+----------+
|        dt|currency|total_transactions|      total_amount|        avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------------+------------------+----------+----------+
|2025-01-01|     AZN|               338| 79560.50999999998|235.38612426035496|    -208.6|   1037.59|
|2025-01-01|     EUR|                66|14668.710000000001|222.25318181818184|     16.44|    688.98|
|2025-01-01|     USD|               137|30555.190000000002|223.03058394160587|   -253.76|    771.66|
|2025-01-02|     AZN|               335| 77816.69999999998|232.28865671641785|   -243.21|   1020.34|
|2025-01-02|     EUR|                82|19577.050000000007|238.74451219512204|     10.28|    677.83|
|2025-01-02|     USD|               139|27819.179999999986| 200.1379856115107|    -74.83|    655.42|
|2025-01-03|     AZN|               354| 81370.61999999998|229.86050847457622|  

In [43]:
# Delta time travel 

delta_old = spark.read \
    .format("delta") \
    .option("versionAsOf", 2) \
    .load(DELTA_GOLD)

print("DELTA - TIME TRAVEL VERSION 2")

delta_old.show()

DELTA - TIME TRAVEL VERSION 2
+----------+--------+------------------+------------------+------------------+----------+----------+
|        dt|currency|total_transactions|      total_amount|        avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------------+------------------+----------+----------+
|2025-01-01|     AZN|               338| 79560.50999999998|235.38612426035496|    -208.6|   1037.59|
|2025-01-01|     EUR|                66|14668.710000000001|222.25318181818184|     16.44|    688.98|
|2025-01-01|     USD|               137|30555.190000000002|223.03058394160587|   -253.76|    771.66|
|2025-01-02|     AZN|               335| 77816.69999999998|232.28865671641785|   -243.21|   1020.34|
|2025-01-02|     EUR|                82|19577.050000000007|238.74451219512204|     10.28|    677.83|
|2025-01-02|     USD|               139|27819.179999999986| 200.1379856115107|    -74.83|    655.42|
|2025-01-03|     AZN|               354| 81370.61999999998|22

In [45]:
# Iceberg snapshot 

spark.sql(f"""
    SELECT *
    FROM {ICEBERG_GOLD}.snapshots
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                     

In [46]:
# Iceberg compare snapshots
snapshots = spark.sql(f"""
    SELECT
        committed_at,
        snapshot_id,
        operation
    FROM {ICEBERG_GOLD}.snapshots
    ORDER BY committed_at
""")

snapshots.show(truncate=False)

+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-09-07 07:55:54.688|6723862207868435361|append   |
|2026-09-07 08:22:05.437|3044808701569443685|append   |
|2026-09-07 08:23:36.365|5871196452952127983|overwrite|
|2026-09-07 08:24:15.099|5742546186733473257|delete   |
|2026-09-07 08:26:02.791|1464907439097166638|append   |
+-----------------------+-------------------+---------+



In [48]:
# Iceberg time travel 
iceberg_old = spark.sql("""
    SELECT *
    FROM iceberg.gold.transaction_summary
    VERSION AS OF 1464907439097166638
""")

iceberg_old.show()

+----------+--------+------------------+------------------+------------------+----------+----------+
|        dt|currency|total_transactions|      total_amount|        avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------------+------------------+----------+----------+
|2026-09-07|     AZN|                20|            3000.0|             150.0|      30.0|     700.0|
|2026-09-07|     USD|                15|            2500.0|            166.67|      20.0|     600.0|
|2025-01-01|     AZN|               338| 79560.50999999998|235.38612426035496|    -208.6|   1037.59|
|2025-01-01|     EUR|                66|14668.710000000001|222.25318181818184|     16.44|    688.98|
|2025-01-01|     USD|               137|30555.190000000002|223.03058394160587|   -253.76|    771.66|
|2025-01-02|     AZN|               335| 77816.69999999998|232.28865671641785|   -243.21|   1020.34|
|2025-01-02|     EUR|                82|19577.050000000007|238.74451219512204|     10.28|  

In [49]:
# Compare delta and Iceberg 

print("===== CURRENT DELTA =====")

spark.read \
    .format("delta") \
    .load(DELTA_GOLD) \
    .show()


print("===== CURRENT ICEBERG =====")

spark.table(ICEBERG_GOLD).show()

===== CURRENT DELTA =====
+----------+--------+------------------+------------------+------------------+----------+----------+
|        dt|currency|total_transactions|      total_amount|        avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------------+------------------+----------+----------+
|2025-01-01|     AZN|               338| 79560.50999999998|235.38612426035496|    -208.6|   1037.59|
|2025-01-01|     EUR|                66|14668.710000000001|222.25318181818184|     16.44|    688.98|
|2025-01-01|     USD|               137|30555.190000000002|223.03058394160587|   -253.76|    771.66|
|2025-01-02|     AZN|               335| 77816.69999999998|232.28865671641785|   -243.21|   1020.34|
|2025-01-02|     EUR|                82|19577.050000000007|238.74451219512204|     10.28|    677.83|
|2025-01-02|     USD|               139|27819.179999999986| 200.1379856115107|    -74.83|    655.42|
|2025-01-03|     AZN|               354| 81370.61999999998|229.86

In [50]:
# Schema evolution for Iceberg (changing column name)

spark.sql("""
    ALTER TABLE iceberg.gold.transaction_summary
    RENAME COLUMN dt TO date
""")

DataFrame[]

In [51]:
spark.table(ICEBERG_GOLD).printSchema()

root
 |-- date: date (nullable = true)
 |-- currency: string (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- avg_amount: double (nullable = true)
 |-- min_amount: double (nullable = true)
 |-- max_amount: double (nullable = true)



In [52]:
# Schema evolution for Delta (adding new column named customer_comments)

spark.sql(f"""
    ALTER TABLE delta.`{DELTA_GOLD}`
    ADD COLUMNS (
        customer_comments STRING
    )
""")


DataFrame[]

In [53]:
spark.read \
    .format("delta") \
    .load(DELTA_GOLD) \
    .printSchema()

root
 |-- dt: date (nullable = true)
 |-- currency: string (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- avg_amount: double (nullable = true)
 |-- min_amount: double (nullable = true)
 |-- max_amount: double (nullable = true)
 |-- customer_comments: string (nullable = true)



In [55]:
#Compare Delta 


delta_result = (
    spark.read
    .format("delta")
    .load(DELTA_GOLD)
)

print("DELTA SCHEMA:")
delta_result.printSchema()

print("DELTA DATA:")
delta_result.show()

DELTA SCHEMA:
root
 |-- dt: date (nullable = true)
 |-- currency: string (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- avg_amount: double (nullable = true)
 |-- min_amount: double (nullable = true)
 |-- max_amount: double (nullable = true)
 |-- customer_comments: string (nullable = true)

DELTA DATA:
+----------+--------+------------------+------------------+------------------+----------+----------+-----------------+
|        dt|currency|total_transactions|      total_amount|        avg_amount|min_amount|max_amount|customer_comments|
+----------+--------+------------------+------------------+------------------+----------+----------+-----------------+
|2025-01-01|     AZN|               338| 79560.50999999998|235.38612426035496|    -208.6|   1037.59|             NULL|
|2025-01-01|     EUR|                66|14668.710000000001|222.25318181818184|     16.44|    688.98|             NULL|
|2025-01-01|     USD|             

In [56]:
# Iceberg compare

iceberg_result = spark.table(ICEBERG_GOLD)

print("ICEBERG SCHEMA:")
iceberg_result.printSchema()

print("ICEBERG DATA:")
iceberg_result.show()

ICEBERG SCHEMA:
root
 |-- date: date (nullable = true)
 |-- currency: string (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- avg_amount: double (nullable = true)
 |-- min_amount: double (nullable = true)
 |-- max_amount: double (nullable = true)

ICEBERG DATA:
+----------+--------+------------------+------------------+------------------+----------+----------+
|      date|currency|total_transactions|      total_amount|        avg_amount|min_amount|max_amount|
+----------+--------+------------------+------------------+------------------+----------+----------+
|2026-09-07|     AZN|                20|            3000.0|             150.0|      30.0|     700.0|
|2026-09-07|     USD|                15|            2500.0|            166.67|      20.0|     600.0|
|2025-01-01|     AZN|               338| 79560.50999999998|235.38612426035496|    -208.6|   1037.59|
|2025-01-01|     EUR|                66|14668.710000000001|222.253

In [57]:
# Compare Delta and Iceberg Schemas

print("DELTA COLUMNS:")
print(delta_result.columns)

print("\nICEBERG COLUMNS:")
print(iceberg_result.columns)

DELTA COLUMNS:
['dt', 'currency', 'total_transactions', 'total_amount', 'avg_amount', 'min_amount', 'max_amount', 'customer_comments']

ICEBERG COLUMNS:
['date', 'currency', 'total_transactions', 'total_amount', 'avg_amount', 'min_amount', 'max_amount']
